# Qwen3.5-0.8B + FlyEmbedding-v3.1 Full Evaluation

This Colab trains the preservation-first FlyEmbedding-v3.1 adapter, validates on Salesforce/WikiText-2, materializes the adapter into a normal Hugging Face Qwen checkpoint, runs fast downstream evaluation, measures speed and GPU memory, creates a model card, and uploads the standalone model to Hugging Face.

FastEval tasks: HellaSwag, PIQA, ARC-Easy, ARC-Challenge, Winogrande, and GSM8K.

The adapter still starts as exact Qwen because the residual scale is zero at initialization.


In [1]:
#@title 1. Setup
import pathlib, subprocess, sys, importlib, torch, os, shutil, json, time

REPO_DIR=pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-U',
    'transformers','accelerate','huggingface_hub','safetensors',
    'datasets','ipywidgets','pandas','matplotlib','lm-eval','tabulate'
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)

SRC_DIR=REPO_DIR/'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0,str(SRC_DIR))
for name in list(sys.modules):
    if name=='tinycenn_lm' or name.startswith('tinycenn_lm.'):
        del sys.modules[name]
importlib.invalidate_caches()

for p in [
    REPO_DIR/'scripts'/'run_qwen35_flyembedding_v31.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flyembedding_v3.py'
]:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)

print('Environment ready')
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
else:
    print('WARNING: switch Colab to a GPU runtime before continuing.')


Environment ready
CUDA: True
GPU: Tesla T4


In [2]:
#@title 2. Configuration
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
RUN_MODE='quick' #@param ['quick','strong']
SEQ_LEN=128 #@param {type:'integer'}

FLY_NODES=256 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}
GRAPH_MIX_INIT=0.05 #@param {type:'number'}
MAX_RESIDUAL_SCALE=0.05 #@param {type:'number'}
LR_CORE=0.00015 #@param {type:'number'}
LR_GATE=0.00030 #@param {type:'number'}

PROBE_EVERY=25 #@param {type:'integer'}
MIN_TOP1=0.99 #@param {type:'number'}
MAX_KL=0.005 #@param {type:'number'}
MAX_EMBEDDING_MSE=0.001 #@param {type:'number'}
EARLY_STOP_TOP1=0.985 #@param {type:'number'}
EARLY_STOP_KL=0.0075 #@param {type:'number'}

FAST_EVAL_LIMIT=50 #@param {type:'integer'}
RUN_BASELINE_EVAL=True #@param {type:'boolean'}

HF_REPO_ID='vtava/Qwen35-0.8B-FlyEmbedding-v31' #@param {type:'string'}
HF_PRIVATE=False #@param {type:'boolean'}

OUTPUT_DIR=REPO_DIR/'results'/'flyembedding_v31_qwen35_08b'
STANDALONE_DIR=REPO_DIR/'results'/'Qwen35-0.8B-FlyEmbedding-v31-standalone'
EVAL_DIR=REPO_DIR/'results'/'flyembedding_v31_fast_eval'

print('HF target:',HF_REPO_ID)
print('FastEval limit per task:',FAST_EVAL_LIMIT)


HF target: vtava/Qwen35-0.8B-FlyEmbedding-v31
FastEval limit per task: 50


In [3]:
#@title 3. Train preservation-first FlyEmbedding-v3.1
cmd=[
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flyembedding_v31.py'),
    '--base-model',BASE_MODEL,
    '--run-mode',RUN_MODE,
    '--seq-len',str(SEQ_LEN),
    '--fly-nodes',str(FLY_NODES),
    '--graph-steps',str(GRAPH_STEPS),
    '--graph-mix-init',str(GRAPH_MIX_INIT),
    '--max-residual-scale',str(MAX_RESIDUAL_SCALE),
    '--lr-core',str(LR_CORE),
    '--lr-gate',str(LR_GATE),
    '--probe-every',str(PROBE_EVERY),
    '--min-top1',str(MIN_TOP1),
    '--max-kl',str(MAX_KL),
    '--max-embedding-mse',str(MAX_EMBEDDING_MSE),
    '--early-stop-top1',str(EARLY_STOP_TOP1),
    '--early-stop-kl',str(EARLY_STOP_KL),
    '--output-dir',str(OUTPUT_DIR)
]
print('='*110); print(' '.join(cmd)); print('='*110)
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait()
if rc:
    raise subprocess.CalledProcessError(rc,cmd)

report=json.loads((OUTPUT_DIR/'report.json').read_text())
print('\nSelected step:',report['selected_step'])
print('Data source:',report['data_source'])
print('Final probe:',json.dumps(report['final_probe'],indent=2))
print('Generation summary:',json.dumps(report['generation_summary'],indent=2))
print('Quality gate:',report['quality_gate_passed'])


/usr/bin/python3 -u /content/TinyCeNN-LM/scripts/run_qwen35_flyembedding_v31.py --base-model Qwen/Qwen3.5-0.8B --run-mode quick --seq-len 128 --fly-nodes 256 --graph-steps 1 --graph-mix-init 0.05 --max-residual-scale 0.05 --lr-core 0.00015 --lr-gate 0.0003 --probe-every 25 --min-top1 0.99 --max-kl 0.005 --max-embedding-mse 0.001 --early-stop-top1 0.985 --early-stop-kl 0.0075 --output-dir /content/TinyCeNN-LM/results/flyembedding_v31_qwen35_08b
DEVICE cuda | dtype torch.bfloat16
Loading Qwen teacher...

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 14981.66it/s]
Loading Qwen + FlyEmbedding-v3.1 student...

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 4838.58it/s]
IDENTITY CHECK {"exact_identity": true, "max_abs_embedding_error": 0.0}

Generating test split: 100%|██████████| 4358/4358 [00:00<00:00, 137559.56 examples/s]

Generating train split: 100%|██████████| 36718/36718 [00:00<00:00, 691191.51 examples/s]

Generating validation split: 100%|██████████| 3760/3760 [0

In [4]:
#@title 4. Materialize Fly adapter into a standard Transformers checkpoint
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flyembedding_v3 import (
    FlyEmbeddingV3Config,
    install_fly_embedding_v3,
    materialize_fly_embedding_v3
)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)

tok=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True)
if tok.pad_token_id is None:
    tok.pad_token=tok.eos_token

model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()

def make_adj(n):
    a=torch.zeros(n,n,dtype=torch.float32)
    for i in range(n):
        a[i,i]=1
        for s in (1,3,7,17):
            a[i,(i+s)%n]=1
            a[i,(i-s)%n]=1
    return (a/a.sum(-1,keepdim=True).clamp_min(1)).to(device)

ckpt=torch.load(OUTPUT_DIR/'fly_embedding_v31_adapter.pt',map_location='cpu')
cfg=FlyEmbeddingV3Config(**ckpt['config'])
install_fly_embedding_v3(model,cfg,make_adj(cfg.fly_nodes))
model.fly_embedding_v3_core.load_state_dict(ckpt['fly_embedding_v3_core'],strict=True)
model.eval()

def generate_ids(m,prompt,max_new_tokens=64):
    text=tok.apply_chat_template([{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True)
    enc=tok(text,return_tensors='pt').to(device)
    with torch.no_grad():
        out=m.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,use_cache=True,pad_token_id=tok.eos_token_id)
    ids=out[0,enc.input_ids.shape[1]:]
    return ids.tolist(),tok.decode(ids,skip_special_tokens=True).strip()

verify_prompts=[
    'What is 17 + 25? Give only the answer.',
    'Write one short sentence about Vienna.',
    'Explain in one sentence what an API is.'
]
before={p:generate_ids(model,p) for p in verify_prompts}

print('Materializing effective embeddings...')
materialize_fly_embedding_v3(model,chunk_rows=4096)
model.eval()
after={p:generate_ids(model,p) for p in verify_prompts}

for p in verify_prompts:
    print('\nUSER:',p)
    print('same tokens:',before[p][0]==after[p][0])
    print('reply:',after[p][1])
    assert before[p][0]==after[p][0]

if STANDALONE_DIR.exists():
    shutil.rmtree(STANDALONE_DIR)
STANDALONE_DIR.mkdir(parents=True,exist_ok=True)
model.save_pretrained(STANDALONE_DIR,safe_serialization=True,max_shard_size='2GB')
tok.save_pretrained(STANDALONE_DIR)
shutil.copy2(OUTPUT_DIR/'report.json',STANDALONE_DIR/'flyembedding_report.json')
shutil.copy2(OUTPUT_DIR/'validation_probes.csv',STANDALONE_DIR/'validation_probes.csv')
shutil.copy2(OUTPUT_DIR/'fly_embedding_v31_adapter.pt',STANDALONE_DIR/'fly_embedding_v31_adapter.pt')

print('Saved standalone checkpoint:',STANDALONE_DIR)
print('tie_word_embeddings:',model.config.tie_word_embeddings)

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Materializing effective embeddings...

USER: What is 17 + 25? Give only the answer.
same tokens: True
reply: 42

USER: Write one short sentence about Vienna.
same tokens: True
reply: Vienna is a historic city known for its rich culture, vibrant nightlife, and stunning architecture.

USER: Explain in one sentence what an API is.
same tokens: True
reply: An API is a set of rules and protocols that allows software applications to communicate with each other.


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

Saved standalone checkpoint: /content/TinyCeNN-LM/results/Qwen35-0.8B-FlyEmbedding-v31-standalone
tie_word_embeddings: False


In [5]:
#@title 5. Verify standalone checkpoint and show generation suite
from transformers import AutoModelForCausalLM, AutoTokenizer

rtok=AutoTokenizer.from_pretrained(STANDALONE_DIR,use_fast=True)
rmodel=AutoModelForCausalLM.from_pretrained(STANDALONE_DIR,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()

def answer(prompt,max_new_tokens=64):
    text=rtok.apply_chat_template([{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True)
    enc=rtok(text,return_tensors='pt').to(device)
    with torch.no_grad():
        y=rmodel.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,use_cache=True,pad_token_id=rtok.eos_token_id)
    return rtok.decode(y[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip()

for i,x in enumerate(report['generation_samples'],1):
    print('\n'+'='*100)
    print(i,'USER:',x['prompt'])
    print('QWEN:',x['qwen_reply'])
    print('FLY :',answer(x['prompt']))
    print('reference jaccard:',round(x['token_jaccard'],3),'passed:',x['passed'])

del rmodel
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]


1 USER: Explain in two sentences why the sky is blue.
QWEN: The sky appears blue because sunlight scatters off the Earth's atmosphere, with shorter wavelengths of blue light being scattered more effectively than longer wavelengths like red and orange.
FLY : The sky appears blue because sunlight scatters off the Earth's atmosphere, with shorter wavelengths of blue light being scattered more effectively than longer ones, while the longer wavelengths of red and orange light travel further and reach our eyes.
reference jaccard: 0.8 passed: True

2 USER: What is 17 + 25? Give only the answer.
QWEN: 42
FLY : 42
reference jaccard: 1.0 passed: True

3 USER: Write one short sentence about Vienna.
QWEN: Vienna is a historic city known for its rich culture, vibrant nightlife, and stunning architecture.
FLY : Vienna is a historic city known for its rich culture, vibrant nightlife, and stunning architecture.
reference jaccard: 1.0 passed: True

4 USER: In one sentence, explain what a residual neur

In [6]:
#@title 6. FastEval on six benchmarks
TASKS='hellaswag,piqa,arc_easy,arc_challenge,winogrande,gsm8k'

if EVAL_DIR.exists():
    shutil.rmtree(EVAL_DIR)
EVAL_DIR.mkdir(parents=True,exist_ok=True)

def run_lm_eval(label,pretrained):
    out=EVAL_DIR/label
    out.mkdir(parents=True,exist_ok=True)
    dtype_name='bfloat16' if dtype==torch.bfloat16 else ('float16' if dtype==torch.float16 else 'float32')
    model_args=f'pretrained={pretrained},dtype={dtype_name},trust_remote_code=True'
    cmd=[
        sys.executable,'-m','lm_eval',
        '--model','hf',
        '--model_args',model_args,
        '--tasks',TASKS,
        '--batch_size','auto',
        '--limit',str(FAST_EVAL_LIMIT),
        '--output_path',str(out)
    ]
    print('\n'+'='*110)
    print(label,':',' '.join(cmd))
    print('='*110)
    rc=subprocess.run(cmd,check=False).returncode
    print(label,'exit code:',rc)
    return rc

run_lm_eval('fly_v31',str(STANDALONE_DIR))
if RUN_BASELINE_EVAL:
    run_lm_eval('qwen_base',BASE_MODEL)



fly_v31 : /usr/bin/python3 -m lm_eval --model hf --model_args pretrained=/content/TinyCeNN-LM/results/Qwen35-0.8B-FlyEmbedding-v31-standalone,dtype=bfloat16,trust_remote_code=True --tasks hellaswag,piqa,arc_easy,arc_challenge,winogrande,gsm8k --batch_size auto --limit 50 --output_path /content/TinyCeNN-LM/results/flyembedding_v31_fast_eval/fly_v31
fly_v31 exit code: 0

qwen_base : /usr/bin/python3 -m lm_eval --model hf --model_args pretrained=Qwen/Qwen3.5-0.8B,dtype=bfloat16,trust_remote_code=True --tasks hellaswag,piqa,arc_easy,arc_challenge,winogrande,gsm8k --batch_size auto --limit 50 --output_path /content/TinyCeNN-LM/results/flyembedding_v31_fast_eval/qwen_base
qwen_base exit code: 0


In [7]:
#@title 7. FastEval comparison table
import pandas as pd
from IPython.display import display

def newest_result(folder):
    candidates=[]
    for p in pathlib.Path(folder).rglob('*.json'):
        try:
            d=json.loads(p.read_text())
            if isinstance(d,dict) and 'results' in d:
                candidates.append((p.stat().st_mtime,p,d))
        except Exception:
            pass
    if not candidates:
        return None,None
    _,p,d=max(candidates,key=lambda x:x[0])
    return p,d

fly_path,fly_eval=newest_result(EVAL_DIR/'fly_v31')
base_path,base_eval=(None,None)
if RUN_BASELINE_EVAL:
    base_path,base_eval=newest_result(EVAL_DIR/'qwen_base')

def pick_metric(task,d):
    r=(d or {}).get('results',{}).get(task,{})
    for k in ['acc_norm,none','acc,none','exact_match,strict-match','exact_match,flexible-extract']:
        if k in r:
            return float(r[k]),k
    for k,v in r.items():
        if isinstance(v,(int,float)) and 'stderr' not in k:
            return float(v),k
    return None,None

rows=[]
for task in TASKS.split(','):
    fv,fk=pick_metric(task,fly_eval)
    bv,bk=pick_metric(task,base_eval)
    rows.append({
        'task':task,
        'metric':fk or bk,
        'Qwen':bv,
        'Fly-v3.1':fv,
        'delta':(fv-bv) if fv is not None and bv is not None else None
    })
bench_df=pd.DataFrame(rows)
display(bench_df)
bench_df.to_csv(STANDALONE_DIR/'fast_eval_comparison.csv',index=False)

if fly_eval:
    (STANDALONE_DIR/'lm_eval_fly_results.json').write_text(json.dumps(fly_eval,indent=2))
if base_eval:
    (STANDALONE_DIR/'lm_eval_qwen_results.json').write_text(json.dumps(base_eval,indent=2))


,task,metric,Qwen,Fly-v3.1,delta
0,hellaswag,"acc_norm,none",0.54,0.54,0.00
1,piqa,"acc_norm,none",0.66,0.66,0.00
2,arc_easy,"acc_norm,none",0.68,0.68,0.00
3,arc_challenge,"acc_norm,none",0.36,0.36,0.00
4,winogrande,"acc,none",0.66,0.64,-0.02
5,gsm8k,"exact_match,strict-match",0.46,0.40,-0.06


In [8]:
#@title 8. Speed, latency and peak GPU memory
from transformers import AutoModelForCausalLM, AutoTokenizer

bench_prompt='Explain in four short sentences how a neural network learns from data.'

def perf_test(label,model_id,repeats=3,new_tokens=64):
    t=AutoTokenizer.from_pretrained(model_id,use_fast=True)
    if t.pad_token_id is None:
        t.pad_token=t.eos_token
    m=AutoModelForCausalLM.from_pretrained(model_id,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()
    text=t.apply_chat_template([{'role':'user','content':bench_prompt}],tokenize=False,add_generation_prompt=True)
    enc=t(text,return_tensors='pt').to(device)
    with torch.no_grad():
        m.generate(**enc,max_new_tokens=8,do_sample=False,use_cache=True,pad_token_id=t.eos_token_id)
    if device.type=='cuda':
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    elapsed=[]; generated=[]
    for _ in range(repeats):
        if device.type=='cuda':
            torch.cuda.synchronize()
        t0=time.perf_counter()
        with torch.no_grad():
            out=m.generate(**enc,max_new_tokens=new_tokens,do_sample=False,use_cache=True,pad_token_id=t.eos_token_id)
        if device.type=='cuda':
            torch.cuda.synchronize()
        elapsed.append(time.perf_counter()-t0)
        generated.append(int(out.shape[1]-enc.input_ids.shape[1]))
    peak=torch.cuda.max_memory_allocated()/1024**3 if device.type=='cuda' else None
    result={
        'model':label,
        'avg_latency_s':sum(elapsed)/len(elapsed),
        'tokens_per_second':sum(generated)/sum(elapsed),
        'peak_gpu_GB':peak
    }
    del m
    if device.type=='cuda':
        torch.cuda.empty_cache()
    return result

perf_df=pd.DataFrame([
    perf_test('Qwen base',BASE_MODEL),
    perf_test('Fly-v3.1 standalone',str(STANDALONE_DIR))
])
display(perf_df)
perf_df.to_csv(STANDALONE_DIR/'performance.csv',index=False)


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

,model,avg_latency_s,tokens_per_second,peak_gpu_GB
0,Qwen base,8.379890,7.637332,1.461949
1,Fly-v3.1 standalone,5.389533,11.874870,1.936559


In [9]:
#@title 9. Create model card and upload to Hugging Face
from huggingface_hub import HfApi, notebook_login

bench=pd.read_csv(STANDALONE_DIR/'fast_eval_comparison.csv') if (STANDALONE_DIR/'fast_eval_comparison.csv').exists() else pd.DataFrame()
perf=pd.read_csv(STANDALONE_DIR/'performance.csv') if (STANDALONE_DIR/'performance.csv').exists() else pd.DataFrame()

def as_md(df):
    if df.empty:
        return 'Not run.'
    return df.to_markdown(index=False)

readme=f"""---
base_model: {BASE_MODEL}
library_name: transformers
pipeline_tag: text-generation
tags:
- qwen
- flyembedding
- residual-adapter
- tinycenn
---

# Qwen3.5-0.8B FlyEmbedding-v3.1

This is a standalone materialized checkpoint produced from {BASE_MODEL} using the TinyCeNN-LM FlyEmbedding-v3.1 identity-preserving residual adapter.

## Method

During training the embedding is:

    e_v3 = e_qwen + alpha * Fly(e_qwen)

Alpha starts at zero, so the model is exactly Qwen at initialization. The best checkpoint is selected only while preservation constraints remain satisfied. The final effective embedding is materialized into a standard Transformers checkpoint.

## Selected checkpoint

- Step: {report.get('selected_step')}
- Data source: {report.get('data_source')}
- Quality gate: {report.get('quality_gate_passed')}
- Top-1 agreement: {report['final_probe']['top1_logit_agreement']:.6f}
- KL to Qwen: {report['final_probe']['teacher_kl']:.6f}
- Validation CE gap: {report['final_probe']['ce_gap']:.6f}
- Training adapter parameters: {report['parameter_stats']['adapter_trainable_params']:,}

## FastEval

{as_md(bench)}

The FastEval results use a limited sample count and are diagnostics, not leaderboard-quality benchmark results.

## Performance

{as_md(perf)}

## Load

    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")
    model = AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}", device_map="auto")

## Important export detail

The source Qwen model ties input and output embeddings. FlyEmbedding modifies input embeddings only, so the standalone materialized checkpoint uses untied input and output weights to preserve the original Qwen LM head. This increases standalone checkpoint size compared with the adapter-only representation.

Project: https://github.com/vtavakkoli/TinyCeNN-LM
"""
(STANDALONE_DIR/'README.md').write_text(readme,encoding='utf-8')
print(readme[:4000])

api=HfApi()
try:
    who=api.whoami()
    print('Already logged in as:',who.get('name',who))
except Exception:
    print('Login to Hugging Face with a write token.')
    notebook_login()
    api=HfApi()
    who=api.whoami()
    print('Logged in as:',who.get('name',who))

api.create_repo(repo_id=HF_REPO_ID,repo_type='model',private=HF_PRIVATE,exist_ok=True)
api.upload_folder(
    folder_path=str(STANDALONE_DIR),
    repo_id=HF_REPO_ID,
    repo_type='model',
    commit_message='Upload FlyEmbedding-v3.1 standalone model with FastEval results'
)
print('Uploaded: https://huggingface.co/'+HF_REPO_ID)


---
base_model: Qwen/Qwen3.5-0.8B
library_name: transformers
pipeline_tag: text-generation
tags:
- qwen
- flyembedding
- residual-adapter
- tinycenn
---

# Qwen3.5-0.8B FlyEmbedding-v3.1

This is a standalone materialized checkpoint produced from Qwen/Qwen3.5-0.8B using the TinyCeNN-LM FlyEmbedding-v3.1 identity-preserving residual adapter.

## Method

During training the embedding is:

    e_v3 = e_qwen + alpha * Fly(e_qwen)

Alpha starts at zero, so the model is exactly Qwen at initialization. The best checkpoint is selected only while preservation constraints remain satisfied. The final effective embedding is materialized into a standard Transformers checkpoint.

## Selected checkpoint

- Step: 50
- Data source: WikiText-2 raw train/validation
- Quality gate: True
- Top-1 agreement: 0.990885
- KL to Qwen: 0.000677
- Validation CE gap: -0.000446
- Training adapter parameters: 525,314

## FastEval

| task          | metric                   |   Qwen |   Fly-v3.1 |   delta |
|:--------

In [10]:
#@title 10. Interactive final test
import ipywidgets as widgets
from IPython.display import display, clear_output

chat_tok=AutoTokenizer.from_pretrained(STANDALONE_DIR,use_fast=True)
if chat_tok.pad_token_id is None:
    chat_tok.pad_token=chat_tok.eos_token
chat_model=AutoModelForCausalLM.from_pretrained(STANDALONE_DIR,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()

prompt_box=widgets.Textarea(
    value='Explain in simple terms why residual adapters are useful.',
    description='Prompt:',
    layout=widgets.Layout(width='100%',height='100px')
)
button=widgets.Button(description='Ask Fly-v3.1',button_style='primary')
out=widgets.Output()

def ask(_):
    q=prompt_box.value.strip()
    if not q:
        return
    text=chat_tok.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True)
    enc=chat_tok(text,return_tensors='pt').to(device)
    with out:
        clear_output()
        with torch.no_grad():
            y=chat_model.generate(**enc,max_new_tokens=160,do_sample=False,use_cache=True,pad_token_id=chat_tok.eos_token_id)
        print(chat_tok.decode(y[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip())

button.on_click(ask)
display(prompt_box,button,out)


Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

Textarea(value='Explain in simple terms why residual adapters are useful.', description='Prompt:', layout=Layo…

Button(button_style='primary', description='Ask Fly-v3.1', style=ButtonStyle())

Output()